Reference: https://maartengr.github.io/BERTopic/index.html#installation

In [34]:
from bertopic import BERTopic
import ssl
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from umap import UMAP
from matplotlib.lines import Line2D
import plotly
import plotly.graph_objects as go
#remove stopwords from the topics list
import nltk
from nltk.corpus import stopwords
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

# Disable SSL verification - because it is annoying
ssl._create_default_https_context = ssl._create_unverified_context

In [35]:
# Download stopwords (if not already downloaded)
nltk.download('stopwords')

# Get the English stopwords
stopwords = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/priyadcosta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
'''
Load the datasets at join them convert the message column into strings. 
'''

df_csop = pd.read_csv('../output/csop_output_chat_level.csv')
df_jury = pd.read_csv('../output/jury_output_chat_level.csv')
df_pgg = pd.read_csv('../output/pgg_output_chat_level.csv')
df_dat = pd.read_csv('../output/DAT_output_chat_level.csv')
df_becker = pd.read_csv('../output/beckerestimation_output_chat_level.csv')
df_gurcay = pd.read_csv('../output/gurcay2015estimation_output_chat_level.csv')
df_csopII = pd.read_csv('../output/csopII_output_chat_level.csv')


df_csop['message'] = df_csop['message'].astype(str)
df_jury['message'] = df_jury['message'].astype(str)
df_pgg['message'] = df_pgg['message'].astype(str)
df_dat['message'] = df_dat['message'].astype(str)
df_becker['message'] = df_becker['message'].astype(str)
df_gurcay['message'] = df_gurcay['message'].astype(str)
df_csopII['message'] = df_csopII['message'].astype(str)

In [37]:
'''
Fit the model based on the message column. Each message is a document 
@param
The list of documents/messages we want to fit the model on
'''
def fit_model_guided(documents,seed_topic_list):
    #umap model initialized to random_state 2 to eliminate stochasticity 
    umap_model = UMAP(n_neighbors=15, n_components=5,min_dist=0.0, metric='cosine', random_state=2)
    model = BERTopic(top_n_words=30,umap_model=umap_model,seed_topic_list=seed_topic_list)
    model.fit_transform(documents)
    return model

In [38]:
def fit_model_unguided(documents):
    #umap model initialized to random_state 2 to eliminate stochasticity 
    umap_model = UMAP(n_neighbors=15, n_components=5,min_dist=0.0, metric='cosine', random_state=42)
    model = BERTopic(top_n_words=30,umap_model=umap_model)
    model.fit_transform(documents)
    return model

In [39]:
'''
Get the top n topics from the model
@param : The BERTopic model, the number of topics we want (This is excluding topic -1)
'''
def get_top_topics(topic_model,n):
    return_list = []
    for i in range(0, n): 
        topic_list = topic_model.get_topic(i)
        return_list.append(topic_list)  
    return return_list  

In [40]:
def reduce_chunks(num_rows, max_num_chunks):
    if (num_rows < max_num_chunks * 2):
        max_num_chunks = int(num_rows / 2)
    if max_num_chunks < 1:
        return 1
    else:
        return max_num_chunks
    

# Assign chunk numbers to the chats within each conversation
def assign_chunk_nums(chat_data, num_chunks):

    # Calculate the total number of rows per conversation
    conversation_lengths = chat_data.groupby('conversation_num').size()

    chunks = conversation_lengths.apply(lambda x: reduce_chunks(x, num_chunks))

    # Calculate the chunk size based on the total number of conversations
    chunk_size = np.ceil(conversation_lengths / chunks) 
    
    for i, group in chat_data.groupby('conversation_num'): # for each group
        chunk_num = 0
        counter = 0

        for chat_id in group.index.values: # iterate over the index values
            chat_data.at[chat_id, 'chunk'] = int(chunk_num)

            counter += 1

            #if counter = 1 for the last row of a group (implies last chunk has one element), and the chunk num > 0, then just make the last one - 1
            if counter == 1 and chunk_num > 0 and chat_id == group.index.values[-1]:
                chat_data.at[chat_id, 'chunk'] = int(chunk_num - 1)

            if counter == chunk_size[i] and chunk_num < chunks[i] - 1: # assign any extras to the last chunk
                chunk_num += 1
                counter = 0    

    return(chat_data)

In [41]:
'''
Divides the conversation into n equal chunks of time and adds a label for each chunk
@param: The dataframe, the desired number of chunks
'''
def create_chunks(df,num_chunks):

    #check if there are timestamps
    if 'timestamp' in df.columns:

        # Convert timestamp column to DateTime format
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        # Calculate the total duration of the conversation
        total_duration = (df['timestamp'].max() - df['timestamp'].min()).total_seconds()
        # total_duration = int(df['duration'][0])

        # Calculate the duration of each chunk
        chunk_duration = total_duration / num_chunks

        if chunk_duration == 0:
            chunk_duration = 1

        # Add a new column for chunk number
        df['chunk'] = -1 

        # Assign the chunk number for each row
        for index, row in df.iterrows():

            #get the timestamp 
            timestamp = row['timestamp']

            #calculate the chunk number
            chunk_number = int(((timestamp - df['timestamp'].min())).total_seconds() / chunk_duration)

            #restrict the range of the chunks from 0 to num_chunks - 1
            if chunk_number >= num_chunks:
                df.at[index, 'chunk'] = num_chunks - 1
            else:
                df.at[index, 'chunk'] = chunk_number
    
    #chunk into parts based on the number of messages
    else:
        assign_chunk_nums(df, num_chunks)

In [42]:
df_temp = df_csop[df_csop['speaker_nickname'] != 'NULL_SPEAKER']
# df_temp = df_csop
convo_min = df_temp['conversation_num'].min()
convo_max = df_temp['conversation_num'].max() + 1

final_df = pd.DataFrame(columns=df_temp.columns)


for i in range(convo_min,convo_max):

    df = df_temp[df_temp['conversation_num'] == i]

    if df is not None:
        create_chunks(df,4)
        final_df = final_df.append(df, ignore_index=True)

df_csop = final_df
print(df_csop)
# final_df.to_csv('csop_with_chunks.csv')

     conversation_num          batch_num          round_num round_index  \
0                   0  26qvN26uKSbi4YagB  449jDgtx63i5j9RcP           5   
1                   0  26qvN26uKSbi4YagB  449jDgtx63i5j9RcP           5   
2                   7  2LB5TwGc9RqGmNGRh  pMY7JBiHvm9w2ipwf           3   
3                   7  2LB5TwGc9RqGmNGRh  pMY7JBiHvm9w2ipwf           3   
4                  14  2NmZsqGiBSjj6MWqS  drtr2Yh4eXzFxRjn4           3   
...               ...                ...                ...         ...   
3473              979  z6ZP2AHyzrNicE8dS  yPMir9ucLQmzFvAof           4   
3474              979  z6ZP2AHyzrNicE8dS  yPMir9ucLQmzFvAof           4   
3475              979  z6ZP2AHyzrNicE8dS  yPMir9ucLQmzFvAof           4   
3476              979  z6ZP2AHyzrNicE8dS  yPMir9ucLQmzFvAof           4   
3477              979  z6ZP2AHyzrNicE8dS  yPMir9ucLQmzFvAof           4   

     task_index complexity type  social_perceptiveness        skill  \
0             3   Moderate  

In [12]:
# nums = 0
# dur = 0
# for i in range(convo_min,convo_max):

#     df = df_temp[df_temp['conversation_num'] == i]

#     if not df.empty:
#         nums = nums + 1
#         dur = dur + df['duration'].iloc[0]

# print(dur/nums)


In [43]:
'''
Extract the SBERT Embeddings from the model
'''
def extract_embeddings(model,df):
    embeddings_matrix = model._extract_embeddings(df['message'].tolist())
    return embeddings_matrix

'''
Get the representative docs for a topic and convert it to a df
'''
def get_rep_docs(model,topic_num):
    data = model.get_representative_docs(topic_num)
    df = pd.DataFrame(data,columns=['rep_docs'])
    return df

'''
Vectorize the documents for the topic
'''
def create_bert_vectors(df,on_column):
    docs = df[on_column].tolist()
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
    return sentence_model.encode(docs, show_progress_bar=False)

In [44]:
'''
Get the embeddings for the top topics across the conversation
'''

def get_embeddings_top_topics(top_topics,model):
    
    embeddings_for_top_topics = []
    
    #for each topic
    for j in range(0,len(top_topics)):
        
        #get the representative documents 
        rep_docs_df = get_rep_docs(model,j)

        #create embeddings of the representative documents for each topic
        topic_embeddings = create_bert_vectors(rep_docs_df,'rep_docs')

        embeddings_for_top_topics.append(topic_embeddings) 
        
    return embeddings_for_top_topics

In [45]:
'''
Get the embeddings for each chunk
'''
def get_embeddings_per_chunk(df,num_chunks):

    chunk_embeddings_list = []

    for i in range(0,num_chunks):
        convo = df[df['chunk'] == i]

        #if the df is empty, i.e there is a pause, create a pseudo 384-dimension embeddings with all values as 0
        if convo.empty:            
            #create n arrays of length 384 (number of dimensions)
            chunk_embeddings = np.zeros((1, 384))

        else:
            chunk_embeddings = create_bert_vectors(convo,'message')
        chunk_embeddings_list.append(chunk_embeddings)
    
    return chunk_embeddings_list

In [46]:
'''
Get the embeddings for each chunk
'''
def get_embeddings_per_convo(df):

    convo_max_embeddings_list = []
    min_chat = df['conversation_num'].min()
    max_chat = df['conversation_num'].max() + 1

    for i in range(min_chat,max_chat):
        convo = df[df['conversation_num'] == i]

        #if the df is empty, i.e there is a pause, create a pseudo 384-dimension embeddings with all values as 0
        if convo.empty:            
            #create n arrays of length 384 (number of dimensions)
            chunk_embeddings = np.zeros((1, 384))

        else:
            chunk_embeddings = create_bert_vectors(convo,'message')

        convo_max_embeddings_list.append(chunk_embeddings)
    
    return convo_max_embeddings_list

In [47]:
#rows - every chunk, cols - topics 
def get_similarity(embeddings_per_chunk,embeddings_top_topics):

    topics_per_chunk = []
    for i in range (0,len(embeddings_per_chunk)):
        
        #calculate the cosine similarity for the topic and the chunk
        embeddings = np.mean(embeddings_per_chunk[i], axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
        embeddings = np.nan_to_num(embeddings, nan=0) #replace any nan vectors with 0
        embeddings = embeddings.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

        topics = []
        
        # 
        for j in range(0,len(embeddings_top_topics)):
            
            topic = np.mean(embeddings_top_topics[j], axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
            topic = np.nan_to_num(topic, nan=0) #replace any nan vectors with 0
            topic = topic.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

            #calculate the cosine similarity betwen the topic embeddings and the chunk embeddings
            cos = cosine_similarity(embeddings,topic)
            topics.append(cos[0][0])

        topics_per_chunk.append(topics)
    
    return topics_per_chunk
    

In [48]:
'''
Remove the stop words from the topic to visualize the topics better
'''
def create_topics_without_stopwords(top_topics):
    cols = []
    for i in range(0,len(top_topics)):
        filtered_words = [word[0] for word in top_topics[i] if word[0].lower() not in stopwords]
        cols.append(filtered_words)

    topic_list = []
    for i in range(0,len(cols)):
        topic = ""
        for j in range(0,len(cols[i])):
            topic = topic + cols[i][j] +", "

        topic_list.append(topic)
    
    return topic_list

In [49]:
'''
#THIS CODE WAS ORIGINALLY WRITTEN BY CHAT GPT, BECAUSE I WAS UNSURE ABOUT THE SYNTAX OF CREATING 
INTERACTIVE VISUALS. I THEN MODIFIED AND SIMPLIFIED IT TO SUIT OUR NEEDS
'''
def plot_with_plotly(df):  

    #Get the headings of the df eg. Convo 0 Topic 0, Convo 0 Topic 1. 
    # I am not changing the original DF so that it is still readable to us when we write it to csv
    convo_columns = df.columns 

    # Extract conversation columns and topics. For e.g, if we have a heading as Convo 0 Topic 1
    #col.split(' ')[-1] gives us the topic number as a string. This extracts the strings of unique topic numbers
    topics = sorted(list(set([col.split(' ')[-1] for col in convo_columns])))

    # Set colors for each topic dynamically - Not yet sure how it exactly works :(
    num_topics = len(topics)
    color_palette = plotly.colors.qualitative.D3
    topic_colors = {topic: color_palette[i % len(color_palette)] for i, topic in enumerate(topics)}

    #I HAVE VERFIED THIS LOGIC BY MANUALLY COMPUTING THE AVERAGES IN AN EXCEL SHEET
    '''
    ABOUT TRACES:

    In Plotly, a "trace" refers to a data object that describes how to visualize a set of data points. 
    It contains information about the type of plot (e.g., scatter, line, bar), the data to be plotted
    (x, y, z coordinates or categorical values), and various styling options such as line color, marker
    shape, and more.
    
    Each trace represents a distinct set of data or a specific aspect of the visualization. For example, 
    if you want to plot multiple lines on the same graph, you would create a separate trace for each line.

    Traces are typically added to a data list and then passed to the `go.Figure` object in Plotly. 
    The `go.Scatter` class is commonly used to create traces for line plots and scatter plots, but there
    are other trace classes available for different types of plots, such as `go.Bar` for bar plots and
    `go.Surface` for 3D surface plots.
    '''

    # Create traces for each conversation and topic
    traces = []
    for col in convo_columns:

        # Splits the original column headsers. For e.g., if the heading is Convo 0 Topic 1,
        convo = col.split(' ')[1] # This will be 0
        topic = col.split(' ')[-1] # This will be 1

        # Set visibility to True for convo 0 and False for others - This means the default value of the drop down is Convo 0
        # visible = True if convo == '0' else False  
        visible = True

        #create a trace object
        trace = go.Scatter(
            x=df.index,
            y=df[col],
            name=col,
            visible=visible,
            line=dict(color=topic_colors[topic], width=1)  # Set color based on topic
        )
        traces.append(trace)

    # Create trace for average of each topic
    avg_traces = []
    for topic in topics:
        '''
        get all the column headers which contain the current topic. For e.g. if the topic is '1', 
        and there are 2 convos. This list contain the headers: Convo 0 Topic 1,Convo 1 Topic 1.
        '''
        topic_columns = [col for col in convo_columns if col.endswith(topic)]

        '''
        Average of Topics Across Convos for each chunk. Axis is 1 as averaging across columns.
        Consider the following example df:

        This df has 3 rows indicating there are 3 chunks of time.
        data = {
            'Convo 0 Topic 0': [0.476381, 0.795204, 0.559577],
            'Convo 0 Topic 1': [0.466211, 0.498702, 0.768713],
            'Convo 1 Topic 0': [0.699774, 0.734668, 0.767569],
            'Convo 1 Topic 1': [0.449561, 0.484407, 0.523591]
        }

        If the current topic is topic 1, the code will average in the following way:

        Average for Chunk 0: (0.466211 + 0.449561) / 2
        Average for Chunk 1: (0.498702 + 0.484407) / 2
        Average for Chunk 2: (0.768713+ 0.523591) / 2

        Verfied by manual calculations in Excel
        '''
        avg_values = np.mean(df[topic_columns], axis=1)

        #create the trace object
        avg_trace = go.Scatter(
            x=df.index,
            y=avg_values,
            name=f'Avg Topic {topic}',
            visible=True,  # Set visibility to True for all average traces
            line=dict(color=topic_colors[topic], width=2)  # Set color based on topic
        )
        avg_traces.append(avg_trace)

    # Add average traces to the list of traces. This list is used to create our drop down
    all_traces = traces + avg_traces

    # Create dropdown options for conversations
    dropdown_options = []
    for convo in range(len(df.columns) // len(topics)):
        option = dict(
            label=f'Convo {convo}',
            method='update',
            args=[{'visible': [False] * len(all_traces)}]  
        )
        for i, trace in enumerate(all_traces):
            if trace.name.startswith(f'Convo {convo}'):
                option['args'][0]['visible'][i] = True  
        option['args'][0]['visible'][-len(topics):] = [True] * len(topics)  # Set visibility to True for average traces
        dropdown_options.append(option)

    # Create layout with dropdown menu
    layout = go.Layout(
        title='Topics Over Time',
        xaxis=dict(
            title='Chunk',
            tickmode='linear',  # Set tick mode to linear to ensure the x-axis i.e chunks contains only integers
            tick0=0,  # Start tick at 0
            dtick=1  # Set tick interval to 1
        ),
        yaxis=dict(title='Cosine Similarity (Topics Vs Chunk Embeddings)'),
        updatemenus=[
            dict(
                buttons=dropdown_options,
                direction='down',
                pad={'r': 10, 't': 10},
                showactive=True,
                x=0.05,
                xanchor='left',
                y=1.15,
                yanchor='top'
            )
        ]
    )

    # Create figure with traces and layout
    fig = go.Figure(data=all_traces, layout=layout)

    # Show the figure
    fig.show()

In [50]:
def plot(df,num_chunks):

    # Set the x-axis tick locations to integers only
    plt.xticks(range(0, num_chunks + 1))

    # Extract the unique topics from the column names. This will extract the topic numbers from 0 to n as strings
    topics = set([col.split(' ')[-1] for col in df.columns])

    lines = []

    #get the list of topics
    for topic in topics:

        # Find columns related to the current topic - e.g. topic 0 in conv0,conv1 etc
        topic_columns = [col for col in df.columns if f'Topic {topic}' in col]

        # Get a color for the current topic
        color = plt.cm.get_cmap('tab10')(int(topic))

        # Plot lines for each column in the current topic
        for column in topic_columns:

            # Plot the line
            line, = plt.plot(df.index, df[column], label=column.split(' ')[0], color=color,alpha=0.5)

            # Get the last index of the DataFrame
            last_index = len(df.index) - 1

            # Get the x and y coordinates for the text label
            x = last_index
            y = df[column].iloc[last_index]

            # Add a text label to the line indicating the conversation number
            plt.text(x, y, f'Convo {column.split(" ")[1]}', color=color, fontsize=10, ha='left', va='bottom')
            lines.append(line)

    # Get the unique colors used in the line graph
    unique_colors = list(set([line.get_color() for line in lines]))

    # create manual tags - NOT THE BEST THING TO DO! NEED TO WORK ON IMPROVING THIS
    manual_tags = ['Topic 1', 'Topic 0']

    # Create custom legend elements
    legend_elements = [Line2D([0], [0], color=color, label=tag) for color, tag in zip(unique_colors, manual_tags)]
    
    # Add the custom legend
    plt.legend(handles=legend_elements)

    # Adding labels and title
    plt.xlabel('Chunk of Time')
    plt.ylabel('Topic Cosine Similarity w.r.t Conversation')
    plt.title('Topics Over Time')

    # Get the current figure
    fig = plt.gcf()

    # Set the new figure size (width, height) in inches
    fig.set_size_inches(8,8)
    
    # Display the chart
    plt.show()

In [51]:
'''
Everything comes together here!
'''

def plot_topics_over_time(df,num_chunks,num_topics,seed_topic_list,approach_type):

    #get the total number of conversations. +1 as the range in python is 0 to n, exclusive of n
    total_convos = df['conversation_num'].max() + 1
    min_convo = df['conversation_num'].min()

    model = None

    if approach_type == 'guided':
        #fit the model
        model = fit_model_guided(df['message'].tolist(),seed_topic_list)
    else:
        model = fit_model_unguided(df['message'].tolist())

    #reduce the topics. We reduce to num_topics + 1 as there will always be a topic -1 with irrelevant topics
    model.reduce_topics(df['message'].tolist(),num_topics+1)

    #get the topics after reduction
    top_topics = get_top_topics(model,num_topics)

    #get the embeddings for the top topics
    embeddings_top_topics = get_embeddings_top_topics(top_topics,model)

    #remove the stopwords from the topics and recreate the topics without the stopwords
    topic_list = create_topics_without_stopwords(top_topics)

    #create an empty df. We will append columns for each Convo-Topic 
    df_for_plotting = None

    #iterate through all the conversations (e.g. 347 in Juries)
    for i in range (min_convo,total_convos):

        print("Convo Number " +str(i))

        #filter the df for a specific conversation number
        df_convo = df[df['conversation_num'] == i]

        #create chunks by dividing the conversation into equal units of time
        create_chunks(df_convo,num_chunks)

        #get the embeddings for each chunk
        embeddings_per_chunk = get_embeddings_per_chunk(df_convo,num_chunks)

        #get the similarity between the topic embeddings and the chunk embeddings
        topics_per_chunk = get_similarity(embeddings_per_chunk,embeddings_top_topics)

        #convert the similarity matrix to a dataframe
        topics_df = pd.DataFrame(topics_per_chunk)

        new_column_headings = []

        #column heading for the current conversation. For e.g. 
        # if it is conversation 131 and we have 2 topics, the headings will be Convo 131 Topic 0, Convo 131 Topic 1
        for j in range(0,len(topic_list)):
            col_name = "Convo "+str(i)+" Topic "+str(j)
            new_column_headings.append(col_name)

        if i == min_convo:#if its the first conversation, we don't need to append the df to an existing df
            df_for_plotting = topics_df
            df_for_plotting.columns = new_column_headings
        else:
            topics_df.columns = new_column_headings
            df_for_plotting = df_for_plotting.join(topics_df)

    #print the top topics
    for i in range(0,len(topic_list)):
        print("Topic " +str(i)+": "+topic_list[i])
    
    #return the df: rows are the chunks, columns are Convo-Topics
    return df_for_plotting

In [ ]:
# #testing on a small portion of the juries dataset
# seed_topic_list = [["hi", "hello", "howdy"],
#                    ["mother", "english", "translate", "husband"],
#                    ["agree", "yes", "vote"]]


# # df_jury = df_jury[df_jury['conversation_num'] < 2]
# df_for_plotting = plot_topics_over_time(df_jury,4,3,seed_topic_list)

In [ ]:
# print(df_for_plotting)
# df_for_plotting.to_csv('df_for_plotting.csv')

In [ ]:
# plot_with_plotly(df_for_plotting)

In [52]:
# load the conversation level data for juries, and classify based on the best predictor score
jury_convo_df = pd.read_csv('../output/jury_output_conversation_level.csv')
# jury_convo_df = jury_convo_df.head(3)

def get_buckets(df,performance_metric,threshold):
    df = df[[performance_metric]]

    #get the mean value of the performace metrix
    mean_value = df[performance_metric].mean()

    lower_threshold = mean_value - (mean_value * threshold)
    upper_threshold = mean_value + (mean_value * threshold)

    # Create a new column to store the classification
    df['label'] = ''

    # Classify the values
    for i, value in enumerate(df['majority_pct']):
        if value < lower_threshold:
            df.loc[i, 'label'] = 'Below Mean'
        elif value > upper_threshold:
            df.loc[i, 'label'] = 'Above Mean'
        else:
            df.loc[i, 'label'] = 'Around Mean'

    return df

In [53]:
def bucket_based_average(df1,df2,num_topics):

    # Create a new DataFrame for aggregation
    df_agg = pd.DataFrame({'Chunk': range(len(df1))})

    # Iterate over the columns in df1 and assign labels based on matching convo numbers
    for col in df1.columns:
        convo = col.split(' ')[1]
        topic = col.split(' ')[-1]
        label = df2['label'][int(convo)]
        label_col = f'{label} Topic {topic}'
        if label_col not in df_agg.columns:
            df_agg[label_col] = 0.0
        df_agg[label_col] += df1[col]

    # Count the number of labels in each bucket
    label_counts = df2['label'].value_counts()
    
    #divide the aggregated values by the number of labels
    for topic in range(0,num_topics):
        topic_col = f'Topic {topic}'
        for label in label_counts.index:
            num_labels = label_counts[label]
            if num_labels > 0:
                df_agg[f'{label} {topic_col}'] /= num_labels 


    # Print the aggregated DataFrame
    df_agg.set_index('Chunk', inplace=True)

    return df_agg

In [54]:
#unpivot the data
def unpivot_data(df,bucket_df,label):

    # Add 'chunk' column with values 0, 1, 2, ...
    df['chunk'] = df.index

    # Reset the index to have a numeric range index
    df.reset_index(drop=True, inplace=True)

    # Unpivot the DataFrame to melt the data
    df_unpivoted = pd.melt(df, id_vars=['chunk'], var_name='convo_topic', value_name='cosine_similarity')

    # Split 'convo_topic' into 'Convo' and 'Topic', and convert them into numbers
    df_unpivoted[['convo', 'topic']] = df_unpivoted['convo_topic'].str.extract(r'Convo (\d+) Topic (\d+)')
    df_unpivoted['convo'] = df_unpivoted['convo'].astype('int')
    df_unpivoted['topic'] = df_unpivoted['topic'].astype('int')

    # Drop the original 'convo_topic' column
    df_unpivoted.drop(columns=['convo_topic'], inplace=True)

    # Reorder the columns so that 'Cosine Similarity' is the first column
    df_unpivoted = df_unpivoted[['cosine_similarity', 'chunk', 'convo', 'topic']]

    # Perform the VLOOKUP-like operation using merge
    result_df = pd.merge(df_unpivoted, bucket_df, left_on='convo',right_index=True, how='left')

    return result_df


# Define the normalization function
def normalize(x):
    return (x - x.mean()) / x.std()


def get_normalized_buckets(df,performance_metric):

    #filter to extract the performance metric
    df = df[[performance_metric]]

    #normalize the performance score
    df['normalized_performance_score'] = df[performance_metric].transform(normalize)

    #get the mean aftet normalization
    mean = df['normalized_performance_score'].mean()

    # Create a new column to store the classification
    df['normalized_label'] = ''

    # Classify the values
    for i, value in enumerate(df['normalized_performance_score']):
        if value < mean:
            df.loc[i, 'normalized_label'] = 'Below Mean'
        else:
            df.loc[i, 'normalized_label'] = 'Above Mean'

    return df

# Define the function to calculate the mean from a bootstrap sample
def bootstrap_mean(data, n_bootstraps=1000):
    valid_data = data.dropna()  # Remove NaN values after normalization
    if len(valid_data) > 1:  # Check if the valid_data has more than one value
        means = []
        for _ in range(n_bootstraps):

            #get an array of random samples
            sample = np.random.choice(valid_data, size=len(valid_data), replace=True)

            #append the mean of random sample to output. 
            means.append(sample.mean())
        # the len will be 1000 due to 1000 bootstraps and each element will be the mean of n random samples (n being the population size)
        return means 
    else: #we cannot bootstrap in this case, as we have only 1 value.
        return np.nan


def normalize_cosine_similarity(df,confidence_interval):

    # Group the DataFrame by 'chunk', 'topic', and 'label_2_buckets'
    grouped = df.groupby(['chunk', 'topic', 'normalized_label'])['cosine_similarity']

    lower_bound = (100 - confidence_interval) / 2
    upper_bound = 100 - lower_bound

    # Calculate the bootstrap confidence intervals for the mean of 'normalized_cosine_similarity' - This will give us the CIs for each chunk-topic-label
    #basically, we will take the 1000 values from above, create a distribution, and get the upper and lower bound based on the confidence interval specified
    confidence_intervals = grouped.apply(bootstrap_mean).dropna().apply(lambda x: np.percentile(x, [lower_bound, upper_bound]))

    #get the aggregated DF. 
    average_df = df.groupby(['chunk', 'topic', 'normalized_label'], as_index=False).agg(cosine_similarity_mean=('cosine_similarity', 'mean'))

    # Create a new DataFrame to store the confidence intervals 
    confidence_intervals_df = pd.DataFrame(confidence_intervals.tolist(), columns=['Lower CI', 'Upper CI'])

    # Concatenate the confidence intervals DataFrame with the original DataFrame - THIS IS A BIT HACKY! TODO - Fix the Hackiness
    result_df = pd.concat([average_df, confidence_intervals_df], axis=1)
    # print(result_df)

    #add the boostrapped results to all the other convos
    return result_df

def convert_convo_nums(df):
    # Use factorize to replace alphanumeric strings with conversation numbers
    df['conversation_num'] = pd.factorize(df['conversation_num'])[0]

In [55]:
def plot3(df,approach_type):
    # Define line styles based on 'normalized_label'
    line_dash_map = {
        ('Above Mean',): 'solid',
        ('Below Mean',): 'dot'
    }

    # Create a color map for different topics
    topic_color_map = {
        topic: f'hsl({360 * topic / df["topic"].nunique()}, 50%, 50%)'
        for topic in df['topic'].unique()
    }

    # Create the figure and subplots
    fig = go.Figure()

    # Iterate over unique topics
    for topic in df['topic'].unique():
        topic_df = df[df['topic'] == topic]

        # Get the color for the current topic
        color = topic_color_map[topic]

        # Iterate over unique normalized_labels for each topic
        for label in topic_df['normalized_label'].unique():
            label_df = topic_df[topic_df['normalized_label'] == label]

            # Create line trace for each topic-chunk-normalized_label combination
            fig.add_trace(go.Scatter(
                x=label_df['chunk'],
                y=label_df['cosine_similarity_mean'],
                mode='lines',
                name=f'Topic {topic} ({label})',
                line=dict(color=color, dash=line_dash_map[(label,)]),
            ))

            # Create vertical bars for confidence intervals at each chunk position
            for i in label_df.index:
                fig.add_shape(
                    go.layout.Shape(
                        type='line',
                        x0=label_df.loc[i, 'chunk'],
                        x1=label_df.loc[i, 'chunk'],
                        y0=label_df.loc[i, 'Lower CI'],
                        y1=label_df.loc[i, 'Upper CI'],
                        line=dict(color=color, dash=line_dash_map[(label,)], width=2),
                    )
                )

        # Calculate the average for the topic-chunk combination
        avg_cosine_similarity = topic_df.groupby('chunk')['cosine_similarity_mean'].mean()

        # Create a thick line trace for the average values
        fig.add_trace(go.Scatter(
            x=avg_cosine_similarity.index,
            y=avg_cosine_similarity,
            mode='lines',
            name=f'Topic {topic} (Average)',
            line=dict(color=color, width=3),
        ))

    if approach_type == 'unguided':
        # Customize layout
        fig.update_layout(
            title='Topics Over Time - Unguided',
            xaxis_title='Chunk',
            yaxis_title='Cosine Similarity Mean',
            legend_title='Topic and Label',
            legend=dict(yanchor='top', y=1.03, xanchor='left', x=1.03),
            xaxis=dict(type='category', tickmode='linear'),
            margin=dict(l=80, r=80, t=100, b=80),  # Adjust the margin around the plot
        )
    else:
        # Customize layout
        fig.update_layout(
            title='Topics Over Time - Guided',
            xaxis_title='Chunk',
            yaxis_title='Cosine Similarity Mean',
            legend_title='Topic and Label',
            legend=dict(yanchor='top', y=1.03, xanchor='left', x=1.03),
            xaxis=dict(type='category', tickmode='linear'),
            margin=dict(l=80, r=80, t=100, b=80),  # Adjust the margin around the plot
        )
        
    # Show the plot
    fig.show()


In [56]:
def plot_after_normalization(df_for_plotting,df,performance_metric,confidence_interval,approach_type):

    bucket_df = get_normalized_buckets(df,performance_metric)

    output = unpivot_data(df_for_plotting,bucket_df,performance_metric)

    bootstrapped = normalize_cosine_similarity(output,confidence_interval)

    plot3(bootstrapped,approach_type)


In [57]:
#ULTIMATE FUNCTION

def train_and_plot(chat_df,seed_topic_list,num_topics,num_chunks,performance_metric,confidence_interval,approach_type):
    if pd.api.types.is_string_dtype(chat_df['conversation_num']):
        print('here')
        convert_convo_nums(chat_df)

    df_for_plotting = plot_topics_over_time(chat_df,num_chunks,num_topics,seed_topic_list,approach_type)

    plot_after_normalization(df_for_plotting,chat_df,performance_metric,confidence_interval,approach_type)

In [ ]:

# plot_after_normalization(df_for_plotting,df_jury,'majority_pct',95)

In [28]:
seed_topics_juries = [['hello', 'hi', 'hey'],
                      ['guy', 'husband', 'asshole', 'mother', 'law', 'language', 'english', 'foreign', 'learn', 'help', 'her', 'polite', 'translate', 'family', 'german', 'spanish', 'selfish', 'immigrant'],
                      ['agree', 'definitely', 'exactly', 'true', 'vote', 'yta', 'nta', 'yes', 'no']]

In [ ]:
# #juries
# train_and_plot(df_jury,seed_topics_juries,3,4,'majority_pct',95,'unguided')
# train_and_plot(df_jury,seed_topics_juries,3,4,'majority_pct',95,'guided')

Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [60]:
#seed topics list
seed_topics_csop = [['move', 'score', 'get', 'change', 'go', 'put','check', 'fix', 'improve', 'do', 'better', 'tweek', 'tweak','points', 'pts', 'money', 'violation', 'maximize', 'goal', 'penalty', 'rule', 'constraint', 'payoff', 'must', 'requirement', 'conflict'],
                   ['looks', 'good', 'job', 'agree', 'submit', 'happy','ok', 'great', 'nice','mess', 'messing', 'ugh', 'stop']]

In [ ]:
# #seed topics list
# seed_topics_csop = [['move', 'score', 'get', 'change', 'go', 'put'],
#                    ['check', 'fix', 'improve', 'do', 'better', 'tweek', 'tweak'],
#                    ['points', 'pts', 'money', 'violation', 'maximize', 'goal', 'penalty', 'rule', 'constraint', 'payoff', 'must', 'requirement', 'conflict'],
#                    ['looks', 'good', 'job', 'agree', 'submit', 'happy'],
#                    ['ok', 'great', 'nice'],
#                    ['mess', 'messing', 'ugh', 'stop']]

In [62]:
#CSOP I
df_csop = df_csop[df_csop['speaker_nickname'] != 'NULL_SPEAKER']
# train_and_plot(df_csop,seed_topics_csop,7,4,'zscore_efficiency',95,'unguided')
train_and_plot(df_csop,seed_topics_csop,3,4,'zscore_efficiency',95,'guided')

Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88433/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [ ]:
#seed topics list
seed_topics_DAT = [['hello', 'morning'],
                   ['good', 'works', 'looks', 'agreed', 'like', 'true', 'satisfy'],
                   ['category', 'animal', 'food', 'plant', 'geological', 'feature', 'desert', 'abstract', 'mountain', 'noun', 'adjective', 'different', 'definition', 'opposite', 'word'],
                   ['toggle', 'submit', 'lock'],
                   ['add', 'replace', 'how', 'about', 'also', 'similar']]

In [ ]:
df_dat = df_dat[df_dat['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_dat,seed_topics_DAT,3,4,'efficiency',95,'unguided')
train_and_plot(df_dat,seed_topics_DAT,3,4,'efficiency',95,'guided')

here
Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [ ]:
#seed topics list
seed_topics_estimation = [[ 'hi', 'hello', 'hey'],
                   ['guess', 'went', 'with', 'said', 'gave', 'put'],
                   ['high', 'low', 'big', 'small', 'off', 'more', 'less', 'realistic'],
                   ['try', 'decide', 'should'],
                   ['agree', 'yes', 'same', 'good']]

In [ ]:
df_becker = df_becker[df_becker['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_becker,seed_topics_estimation,5,4,'mean_post_discussion_error_pct',95,'unguided')
train_and_plot(df_becker,seed_topics_estimation,5,4,'mean_post_discussion_error_pct',95,'guided')

here
Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [ ]:
seed_topics_gurcay = [[ 'burj khalifa', 'empire state', 'building'],
                      ['tall', 'height', 'feet','inches','inch'],
                      ['german shepherd', 'irish wolfhound','dog', 'feet'],
                      ['down', 'standing','sideways', 'up'],
                      ['marriage', 'divorce','first marriage', 'katie holmes','tom cruise','kim kardashian','heartbreak'],
                      ['recession', 'money','sideways', 'up'],
                      ['alaska', 'hawaii','north', 'south','wide','midsize','large'],
                      ['thousand', 'millions','billions', 'miles'],
                      ['labor force', 'poverty','exempt', 'child labor']
                    ]

In [ ]:
df_gurcay = df_gurcay[df_gurcay['speaker_nickname'] != 'NULL_SPEAKER']

train_and_plot(df_gurcay,seed_topics_gurcay,3,4,'mean_post_discussion_error_pct',95,'guided')

Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Topic 0: nan, think, like, 50, 20, 10, high, yeah, 30, ok, around, 15, say, 
Topic 1: catholic, dog, pet, pets, fish, dogs, christian, great, christians, protestant, people, catholics, feet, many, danes, dane, 
Topic 2: closet, trapped, count, poop, ofver, hel, one, rkelly, lmao, longer, hour, thinking, video, lol, im, say, like, 


In [ ]:
train_and_plot(df_gurcay,seed_topics_gurcay,3,4,'mean_post_discussion_error_pct',95,'unguided')

Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Topic 0: think, like, 50, 20, 10, high, around, yeah, 30, say, ok, 15, 
Topic 1: nan, wharton, finance, whartonites, architects, economist, econ, come, kids, economists, accountantsgt, gooby, dudes, classes, ze, help, subscribers, astronomy, semester, pls, majors, house, took, room, guy, 
Topic 2: catholic, christian, christians, protestant, catholics, majority, christianity, church, percentage, religion, many, minority, think, mostly, people, us, yeah, usa, 


In [ ]:
#CSOP II
df_csopII = df_csopII[df_csopII['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_csopII,seed_topics_csop,7,4,'efficiency',95,'unguided')
train_and_plot(df_csopII,seed_topics_csop,7,4,'efficiency',95,'guided')

here
Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [ ]:
#seed topics list
seed_topics_pgg = [[ 'hi', 'hayy', 'hello','howdy'],
                   [ 'coins', 'bonus', 'maximum','payout','contribution'],
                   ['guess', 'round','surprise','number','game'],
                   ['high', 'low', 'big', 'small', 'off', 'more', 'less', 'realistic']]

In [ ]:
df_pgg = df_pgg[df_pgg['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_pgg,seed_topics_pgg,4,4,'score',95,'unguided')
train_and_plot(df_pgg,seed_topics_pgg,4,4,'score',95,'guided')

Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Topic 0: 20, im, round, time, rounds, put, lol, go, everyone, get, 15, 
Topic 1: moose, monkey, gorilla, come, except, stop, money, sharing, deduct, think, motives, troll, mooching, sense, want, 
Topic 2: bird, red, duck, lying, chick, ha, angry, plant, chicken, lol, game, bad, pet, lied, theyre, yellow, lunch, idk, sentient, interruption, interface, innocent, play, lies, oh, 
Topic 3: arrow, button, press, click, give, guys, increases, works, wrong, oops, page, pot, confusing, confused, clicking, change, buttons, til, shared, wont, goes, accident, actually, always, done, contribution, 


Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Topic 0: 20, round, im, rounds, get, put, hi, lol, time, 10, everyone, 15, going, 
Topic 1: monkey, bird, red, duck, lying, bad, chick, chicken, gorilla, ha, plant, angry, game, lol, stop, sharing, lied, selfish, lies, sense, 
Topic 2: moose, except, come, deduct, slow, person, slacking, 200, want, dropped, sucks, cry, cooperate, buddy, afk, shared, tell, times, isnt, almost, hes, 
Topic 3: arrow, button, press, click, give, guys, increases, works, wrong, oops, page, pot, confusing, confused, clicking, change, buttons, til, contribution, shared, wont, accident, goes, actually, done, always, 


In [ ]:
#CSOP - 3 TOPICS
df_csop = df_csop[df_csop['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_csop,seed_topics_csop,3,4,'efficiency',95,'unguided')

Convo Number 0
Convo Number 1
Convo Number 2
Convo Number 3
Convo Number 4
Convo Number 5
Convo Number 6
Convo Number 7
Convo Number 8
Convo Number 9
Convo Number 10
Convo Number 11
Convo Number 12
Convo Number 13
Convo Number 14
Convo Number 15
Convo Number 16
Convo Number 17
Convo Number 18
Convo Number 19
Convo Number 20
Convo Number 21
Convo Number 22
Convo Number 23
Convo Number 24
Convo Number 25
Convo Number 26
Convo Number 27
Convo Number 28
Convo Number 29
Convo Number 30
Convo Number 31
Convo Number 32
Convo Number 33
Convo Number 34
Convo Number 35
Convo Number 36
Convo Number 37
Convo Number 38
Convo Number 39
Convo Number 40
Convo Number 41
Convo Number 42
Convo Number 43
Convo Number 44
Convo Number 45
Convo Number 46
Convo Number 47
Convo Number 48
Convo Number 49
Convo Number 50
Convo Number 51
Convo Number 52
Convo Number 53
Convo Number 54
Convo Number 55
Convo Number 56
Convo Number 57
Convo Number 58
Convo Number 59
Convo Number 60
Convo Number 61
Convo Number 62
Co

/var/folders/8j/rzq1_zj938vgp1cqknfrmc0m0000gn/T/ipykernel_88823/2806592424.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



In [ ]:
df_csopII = df_csop[df_csop['speaker_nickname'] != 'NULL_SPEAKER']
train_and_plot(df_csop,seed_topics_csop,3,4,'efficiency',95,'unguided')

In [29]:
#Validation

def validation(df,num_topics,top_chats_num):
    if pd.api.types.is_string_dtype(df['conversation_num']):
        print('converting convo nums')
        convert_convo_nums(df)
    
    #train the model
    model = fit_model_unguided(df['message'].tolist())

    #generate a list of top topics
    top_topics = get_top_topics(model,num_topics)

    #generate the embeddings for the top topics
    topic_embeddings = get_embeddings_top_topics(top_topics,model)

    #generate embeddings for each chat
    convo_embeddings = get_embeddings_per_convo(df)
    
    top_chats = []
    bottom_chats = []

    #for each topic
    for i in range(0,len(topic_embeddings)):

        topic_embedding = topic_embeddings[i]

        topic = np.mean(topic_embedding, axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
        topic = np.nan_to_num(topic, nan=0) #replace any nan vectors with 0
        topic = topic.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)
    
        data = []
        
        #for each convo
        for j in range(0,len(convo_embeddings)):

            convo_embedding = convo_embeddings[j]

            convo = np.mean(convo_embedding, axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
            convo = np.nan_to_num(convo, nan=0) #replace any nan vectors with 0
            convo = convo.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

            #compute cosine similarity between the topic and the chat
            cosine_sim = cosine_similarity(convo,topic)

            convo_sim_tuple = (j,cosine_sim[0][0]) #chat - cosine sim

            #store the chat-cosine similarity as a tuple
            data.append(convo_sim_tuple)

        #sort in descending order of cosine similarity
        sorted_data = sorted(data, key=lambda x: x[1], reverse=True)

        #get the top and bottom chats numbers
        top_n_tuples = sorted_data[:top_chats_num]
        bottom_n_tuples = sorted_data[-top_chats_num:]

        # Extract the value at the desired index from each top tuple
        top_chats_topic = [t[0] for t in top_n_tuples]
        bottom_chats_topic = [t[0] for t in bottom_n_tuples]

        top_chats.append(top_chats_topic)
        bottom_chats.append(bottom_chats_topic)


    #remove the stopword from the topics
    topics_without_stopwords = create_topics_without_stopwords(top_topics)

    print("Topic-Wise Top Chats " + str(top_chats))
    print("Topic-Wise Bottom Chats " + str(bottom_chats))

    #print the top topics and the chats which they appear in
    for i in range(0,len(topics_without_stopwords)):
        print("Topic Number "+str(i)+": "+topics_without_stopwords[i])
        print("TOP CONVOS")
        
        for j in range(0,len(top_chats)):
            print("Convo Number "+str(top_chats[i][j]))
            temp = df[df['conversation_num'] == top_chats[i][j]]
            print(temp['message'])
        
        print("----------------------")

        print("BOTTOM CONVOS")
                
        for j in range(0,len(bottom_chats)):
            print("Convo Number "+str(bottom_chats[i][j]))
            temp = df[df['conversation_num'] == bottom_chats[i][j]]
            print(temp['message'])

In [30]:
#Validation

def validation_chat_level(df,num_topics,top_chats_num):
    if pd.api.types.is_string_dtype(df['conversation_num']):
        print('converting convo nums')
        convert_convo_nums(df)
    
    #train the model
    model = fit_model_unguided(df['message'].tolist())

    #generate a list of top topics
    top_topics = get_top_topics(model,num_topics)

    #generate the embeddings for the top topics
    topic_embeddings = get_embeddings_top_topics(top_topics,model)

    #generate embeddings for each chat
    convo_embeddings = get_embeddings_per_convo(df)
    
    top_chats = []
    bottom_chats = []

    #for each topic
    for i in range(0,len(topic_embeddings)):

        #create a new column for the topic cosine similarity
        df[f'Topic_{i}'] = None
        counter = 0

        topic_embedding = topic_embeddings[i]

        topic = np.mean(topic_embedding, axis=0) #get the average vectors for the chunk(if there are 4 docs, we get the average of 4 docs for each of the 384 dimensions)
        topic = np.nan_to_num(topic, nan=0) #replace any nan vectors with 0
        topic = topic.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)
    
        
        #for each convo
        for j in range(0,len(convo_embeddings)):

            convo_embedding = convo_embeddings[j]


            #for each chat in each convo
            for k in range (0,len(convo_embedding)):

                convo = convo_embedding[k]
                convo = np.nan_to_num(convo, nan=0) #replace any nan vectors with 0
                convo = convo.reshape(1, -1) #reshape to a 2D matrix (needed for the cosine similarity function)

                #compute cosine similarity between the topic and the chat
                cosine_sim = cosine_similarity(convo,topic)

                df.loc[counter,f'Topic_{i}'] = cosine_sim[0]

                counter = counter + 1

In [31]:
validation_chat_level(df_jury,2,2)

In [33]:
df_jury.to_csv('cosine_chat_jury.csv')

In [ ]:
# temp = df_jury[df_jury['conversation_num'] < 5]
# validation(df_jury,2,2)

Topic-Wise Top Chats [[236, 136], [46, 311]]
Topic-Wise Bottom Chats [[271, 71], [59, 230]]
Topic Number 0: german, speak, speaking, germany, english, american, mom, mother, husband, time, around, spoke, conversations, guy, speaks, comfortable, could, weekend, spouse, talk, 
TOP CONVOS
Convo Number 236
9554                                                i dom
9555    this one has a gray area but in general i thin...
9556                i think he should be speaking english
9557    i think she should be speaking english out of ...
9558                   he is a guest at his mothers house
9559    i think he should be speaking english and then...
9560    he is staying in his mothers house and it woul...
9561    i dont think the person is an intentional assh...
9562    it doesnt matter if his husband is uncomfortab...
9563    i dont see a problem with her speaking german ...
9564    everyone present should speak and understand e...
9565    why cant the guy have some consideration for h...
